# MUSTER 그래프 200·400 첫 답 성능 비교

KnowMe 고정 300문항 중 확정된 변경 문항 80개를 비교합니다. 동일 문항·동일 문맥·동일 판독 조건만 합쳐 199문맥을 실제 평가합니다(재시도 전 reader 199회 + judge 199회). 전체 벤치의 절대 점수로 표시하지 않습니다.

실행 소유자는 Codex Release이며 새 graph_quality_20261116 원장의 전체 내부 상한은 $5입니다. 기존 rr82 원장을 재사용하지 않습니다. CPU 런타임에서 Run all 하십시오. 이 노트북은 API를 Colab에서만 호출하며 GPU 모델은 만들지 않습니다.

별도로 받은 비공개 ZIP을 업로드하면 해시·입력·정답 분리·독립 예산 상한을 확인합니다. 정답 파일이 든 ZIP은 GitHub/PyPI/HF에 공개하지 마십시오. Colab secret/env의 OPENROUTER_API_KEY를 사용하며, 없으면 키가 없다고만 표시합니다.

In [ ]:
import os, sys, json, hashlib, zipfile, stat, subprocess
from pathlib import Path, PurePosixPath
import google.colab
from google.colab import files, drive, userdata
print("Colab 전용 새 원장: graph_quality_20261116 | owner Codex Release | 전체 내부 상한 $5")
print("준비: 300개 기준 문항, 변경 80문항, 실제 199문맥, 재시도 전 API 398회")
print("현재 새 모델/API 실행 0회. 입력 입장 통과 뒤에만 실행합니다.")

In [ ]:
EXPECTED_ZIP_SHA256 = "e32a8480aea59e7b5a1172953bd5ea5486414b1a0c39636ba837a4fbc6fd1990"
print("비공개 MUSTER_graph_quality_knowme300_first_wave_private_v2.zip 한 개를 선택하세요.")
uploaded = files.upload()
assert len(uploaded) == 1, "ZIP 한 개만 필요합니다."
zip_name, raw_zip = next(iter(uploaded.items()))
assert hashlib.sha256(raw_zip).hexdigest() == EXPECTED_ZIP_SHA256, "비공개 ZIP SHA가 다릅니다."
root = Path("/content/muster_graph_quality_20261116")
root.mkdir(exist_ok=True)
archive = root / "private_inputs.zip"
archive.write_bytes(raw_zip)
package = root / "package"
package.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.infolist():
        p = PurePosixPath(m.filename)
        assert not p.is_absolute() and ".." not in p.parts and not stat.S_ISLNK(m.external_attr >> 16), "잘못된 ZIP 경로"
    z.extractall(package)
del uploaded, raw_zip
print("비공개 ZIP SHA 확인 통과.")

In [ ]:
venv = root / "venv"
if not (venv / "bin/python").exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(venv)], check=True)
python = str(venv / "bin/python")
subprocess.run([python, "-m", "pip", "install", "--quiet", "-r", str(package / "requirements.txt")], check=True)
print("격리된 API 실행 환경을 준비했습니다. GPU·검색 모델 호출은 없습니다.")

In [ ]:
import signal

def run_visible(cmd, env=None):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        print("현재 노트북이 시작한 자식 실행만 중지합니다. 원장의 미확정 호출은 자동 재전송하지 않습니다.", flush=True)
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
        raise
    finally:
        if process.stdout is not None:
            process.stdout.close()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

drive.mount("/content/drive")
out = Path("/content/drive/MyDrive/MUSTER_graph_quality_20261116/knowme300_wave1")
command = [python, "-B", str(package / "graph_quality_wave.py"), "--package", str(package), "--out", str(out), "--workers", "12"]
run_visible(command + ["--mode", "plan"])
key = os.environ.get("OPENROUTER_API_KEY")
if not key:
    for secret_name in ("OPENROUTER_API_KEY", "OPENROUTER_KEY"):
        try:
            key = userdata.get(secret_name)
        except Exception:
            key = None
        if key:
            break
if not key:
    print("BLOCKED_API_KEY_MISSING: Colab secret/env에 OpenRouter 키가 없습니다. 실제 호출 0회입니다.")
else:
    run_env = os.environ.copy()
    run_env["OPENROUTER_API_KEY"] = key
    print("입장 통과. 단일 owner/새 원장으로 실행합니다. 30초마다 완료 호출·총량·실청구·실패/차단을 출력합니다.")
    run_visible(command + ["--mode", "run"], env=run_env)
    del run_env
del key

In [ ]:
summary = out / "SUMMARY.json"
if summary.exists():
    print(json.dumps(json.loads(summary.read_text()), ensure_ascii=False, indent=2))
else:
    print("아직 완료 SUMMARY가 없습니다. 위의 PROGRESS 또는 BLOCKED 사유를 확인하세요.")
print("결과 위치:", out)